Test Lambda Function

In [2]:
import pandas; print("Pandas", pandas.__version__)

Pandas 2.2.2


In [3]:
##########
# Sagemaker Setup
##########
import boto3
import sagemaker
from sagemaker import get_execution_role
import sys
import IPython

role = get_execution_role()
sess = sagemaker.Session()
print("Session = {}".format(sess))
region = boto3.session.Session().region_name
print("Region = {}".format(region))
sm = boto3.Session().client('sagemaker')
#Define S3 Bucket
rawbucket= sess.default_bucket() # Alternatively you can use our custom bucket here. 
print("Bucket = {}".format(rawbucket))

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
Session = <sagemaker.session.Session object at 0x7ff5425bae00>
Region = us-east-1
Bucket = sagemaker-us-east-1-719805443631


In [ ]:
import os
import json
import pandas as pd
import pickle
from io import BytesIO
import datetime
import time

runtime_client = boto3.client('runtime.sagemaker')
EndpointName='bi-oead-prob-sm-xgboost-rt'
Preprocess='preprocessing-rt.csv'
#sess = sagemaker.Session()
bucket= sess.default_bucket()
prefix = 'oead-prob' # use this prefix to store all files
modelprefix = 'oead-prob/model/'
dataprefix = prefix + '/train_data/'
sess = sagemaker.Session()

event = {
  "dow": 3,
  "ageGroup": 2,
  "studLevel": 3,
  "stuH": "4-8",
  "country_iso": "111",
  "enrollment": "sMx",
  "deltaDays": -30,
  "deltaHours": -1,
  "hourOfDay": 9,
  "minuteOfHour": 10,
  "isWeekend": "false",
#  "isOrientation": 0,
#  "isHoliday": 0,
#  "isPrivate": 0,
  "book_date":"2023-12-20" ,
  "native_language":"es",
  "class_type":"group" 
}

df_holiday = pd.read_csv('s3://{}/{}'.format(rawbucket,dataprefix)+'lk_holiday.csv')

def lambda_handler(event, context):
    event = {
                "deltaDays": -2,
                "deltaHours": 13,
                "dow": 1,
                "hourOfDay": 4,
                "minuteOfHour": 0,
                "isWeekend": "false",
                "ageGroup": 2,
                "studLevel": 2,
                "isOrientation": 0,
                "stuH": "4-6",
                "isHoliday": 0,
                "country_iso": "CL",
                "enrollment": "sMx",
                "isPrivate": 0,
                "isHolidayPre": 0,
                "isHolidayPost": 0,
                "native_language": "es",
                "class_type": "group",
                "book_date":"2024-01-08"
            #    "J": 0,
            #    "current_prob": 0.741,
            #    "current_prediction": 1,
            #    "slotMIA": "2024-01-08 04:00:00.000",
            #    "reservationMIA": "2024-01-06 17:14:28.000",
            #    "xgboost_pred": 0,
            #    "xgboost_prob": 0.14180265367031097

              }
                
    current_time = datetime.datetime.now()
    start_no_pc = time.time()
    #load preprocess template
    preprocess_template_path = 's3://{}/{}'.format(bucket,modelprefix) 
    #print(preprocess_template_path)
    preprocessing_load = pd.read_csv(preprocess_template_path+Preprocess)
    #preprocessing_load = preprocessing_load.drop(['Unnamed: 0'],axis=1)
    
    df_categorical = pd.DataFrame()
    #categorical
    categorical_features = ['stuH', 'country_iso', 'enrollment','native_language','class_type']
    remainder_features = ['deltaDays', 'deltaHours','dow','hourOfDay','minuteOfHour','isWeekend', 'ageGroup', 'studLevel',
                          'isHoliday','isHolidayPre','isHolidayPost']
    
    event['isHolidayPre']=0
    event['isHolidayPost']=0
    
    holiday = df_holiday.loc[(df_holiday['date'] == event['book_date']) & (df_holiday['country_code'] == event['country_iso'])]
    if holiday.empty==False:
        event['isHoliday']=1
    else:
        event['isHoliday']=0
    
    
    date_post=datetime.datetime.strptime(event['book_date'], "%Y-%m-%d")+ datetime.timedelta(days=1)
    date_post=df_holiday.loc[(df_holiday['date'] == date_post.strftime("%Y-%m-%d")) & (df_holiday['country_code'] == event['country_iso'])].date

    if date_post.empty==False:
        event['isHolidayPost']=1
    else:
        event['isHolidayPost']=0

    date_pre=datetime.datetime.strptime(event['book_date'], "%Y-%m-%d")+ datetime.timedelta(days=-1)
    date_pre=df_holiday.loc[(df_holiday['date'] == date_pre.strftime("%Y-%m-%d")) & (df_holiday['country_code'] == event['country_iso'])].date

    if date_pre.empty==False:
        event['isHolidayPre']=1
    else:
        event['isHolidayPre']=0
    
    del event['book_date']
    
    event['isWeekend'] = int(event['isWeekend'] == 'true')
    #to deprecate
    #event['isOrientation']=0
    #event['isPrivate']=0
    #end
    event = pd.DataFrame([event])

    event_ohe = event[categorical_features].copy()
    event_remainder = event[remainder_features].copy()

    #create dummy variable for each categorical feature
    for x in categorical_features:
        df = pd.get_dummies(event_ohe[[x]],columns=[str(x)],prefix=str(x)) 
        event_ohe.drop(columns=[x],axis=1,inplace=True)
        df_categorical = df.join(df_categorical)


    #combine categorical features with ohe and remainder features in 1 dataframe
    event_preprocessed = event_remainder.combine_first(df_categorical.astype(int))
    event_preprocessed = event_preprocessed.reset_index(drop=True)
    #event_preprocessed.to_csv('event_preprocessed.csv', index=False)
    #combine even_preprocessed with the rest of ohe columns from trainning dataset
    #predict = event_preprocessed.join(preprocessing_load.drop(columns=event_preprocessed.columns,axis=1))
    
    predict = preprocessing_load.merge(event_preprocessed, how='right')
    predict = predict.fillna(0)
    predict.to_csv('predict.csv')
    data = predict.to_csv(header=False,index=False)
    
    #payload = '2,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,-1,6,0.0,0.0,0.0,0.0,0.0,0.0,1,11,1,0,0,0,30,1,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1'
    #payload='2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1'
    #payload = data
    payload='2,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2,13,1,0.0,0.0,0.0,0.0,0.0,0.0,1,4,0,0,0,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,2'
    print("payload:" + payload)  
    print(EndpointName)
    response = runtime_client.invoke_endpoint(EndpointName=EndpointName,
                                       ContentType='text/csv',
                                       Body=payload)
    #print(response)
    #result = json.loads(response['Body'].read().decode())
    #result = response['Body'].read().decode('ascii')
    #result=0
    end_time = time.time() - start_no_pc
    print(EndpointName)
    return result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time

In [23]:
result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)

payload:2,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2,13,1,0.0,0.0,0.0,0.0,0.0,0.0,1,4,0,0,0,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,2
bi-oejr-prob-sm-xgboost-rt


ModelError: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received server error (500) from primary with message "Unable to load model: Model at /opt/ml/model cannot be loaded:
invalid load key, '{'.
[18:04:31] /workspace/src/common/json.cc:413: Expecting: """, got: "L ", around character: 1
    {L". See https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logEventViewer:group=/aws/sagemaker/Endpoints/bi-oejr-prob-sm-xgboost-rt in account 719805443631 for more information.

In [15]:
result

0.14180275797843933

In [ ]:
payload='2,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2,13,1,0.0,0.0,0.0,0.0,0.0,0.0,1,4,0,0,0,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,2'
print("payload:" + payload)  
response = runtime_client.invoke_endpoint(EndpointName='bi-oead-prob-sm-xgboost-rt',
                                       ContentType='text/csv',
                                       Body=payload)

payload:2,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2,13,1,0.0,0.0,0.0,0.0,0.0,0.0,1,4,0,0,0,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,2


ModelError: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received server error (500) from primary with message "Unable to load model: Model at /opt/ml/model cannot be loaded:
invalid load key, '{'.
[18:04:39] /workspace/src/common/json.cc:413: Expecting: """, got: "L ", around character: 1
    {L". See https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logEventViewer:group=/aws/sagemaker/Endpoints/bi-oejr-prob-sm-xgboost-rt in account 719805443631 for more information.

Test Latency

In [ ]:
import time
import numpy as np
print("Testing cold start for serverless inference with PC vs no PC")

pc_times = []
non_pc_times = []

# ~50 minutes
for i in range(5):
    time.sleep(600)
    start_pc = time.time()
    pc_response = runtime_client.invoke_endpoint(
        EndpointName='bi-oead-prob-sm-xgboost-pc',
        Body='2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1',
        ContentType="text/csv",
    )
    end_pc = time.time() - start_pc
    pc_times.append(end_pc)

    start_no_pc = time.time()
    response = runtime_client.invoke_endpoint(
        EndpointName='bi-oead-prob-sm-xgboost',
        Body='2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1',
        ContentType="text/csv",
    )
    end_no_pc = time.time() - start_no_pc
    non_pc_times.append(end_no_pc)

pc_cold_start = np.mean(pc_times)
non_pc_cold_start = np.mean(non_pc_times)

print("Provisioned Concurrency Serverless Inference Average Cold Start: {}".format(pc_cold_start))
print("On Demand Serverless Inference Average Cold Start: {}".format(non_pc_cold_start))

Testing cold start for serverless inference with PC vs no PC
Provisioned Concurrency Serverless Inference Average Cold Start: 0.34101076126098634
On Demand Serverless Inference Average Cold Start: 0.07101831436157227


Stress Test / Cold Starts

In [20]:
%%time
import time
import numpy as np
print("Testing")

pc_times = []
non_pc_times = []
start = time.time()
# ~50 minutes
for i in range(5):
    time.sleep(60)
    print("Request:", i)
    start_no_pc = time.time()
    
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    result,data,event_preprocessed,predict,payload,preprocessing_load,response,end_time=lambda_handler(event,0)
    print("---Prob:", result, " time:", end_time * 1000, " ms")
    
    end_no_pc = time.time() - start_no_pc    
    print("Invokation in ms: {}".format((end_no_pc * 1000 )))
    non_pc_times.append(end_no_pc)

non_pc_cold_start = np.mean(non_pc_times)

print("On Demand Serverless Inference Average Cold Start: {}".format(non_pc_cold_start * 1000))
total = time.time() - start
print("Total:",total * 1000)

Testing
Request: 0
---Prob: 0.9689412713050842  time: 285.2489948272705  ms
---Prob: 0.9689412713050842  time: 124.43900108337402  ms
---Prob: 0.9689412713050842  time: 109.81011390686035  ms
---Prob: 0.9689412713050842  time: 88.76585960388184  ms
---Prob: 0.9689412713050842  time: 93.98794174194336  ms
---Prob: 0.9689412713050842  time: 108.72817039489746  ms
---Prob: 0.9689412713050842  time: 110.5949878692627  ms
---Prob: 0.9689412713050842  time: 83.22620391845703  ms
Invokation in ms: 1007.4639320373535
Request: 1
---Prob: 0.9689412713050842  time: 150.00367164611816  ms
---Prob: 0.9689412713050842  time: 142.24982261657715  ms
---Prob: 0.9689412713050842  time: 106.64248466491699  ms
---Prob: 0.9689412713050842  time: 142.26055145263672  ms
---Prob: 0.9689412713050842  time: 108.54268074035645  ms
---Prob: 0.9689412713050842  time: 107.76305198669434  ms
---Prob: 0.9689412713050842  time: 99.98679161071777  ms
---Prob: 0.9689412713050842  time: 147.87006378173828  ms
Invokation 

Invoke Lambda

In [ ]:
response_data = []
# Create a Boto3 client for AWS Lambda
lambda_client = boto3.client('lambda')
 
# Define a function to invoke the Lambda function with a given payload
def invoke_lambda(payload):
    "Method to invoke the lambda function"
    # Define the Lambda function name and its payload data
    function_name = 'bi_ml_oead-prob'
 
    # Invoke the Lambda function with the payload data
    response = lambda_client.invoke(
        FunctionName=function_name,
        Payload=payload,
        InvocationType='Event'
    )
    response_data.append(response)

In [18]:
payload = {
  "dow": 3,
  "ageGroup": 2,
  "studLevel": 3,
  "stuH": "4-8",
  "country_iso": "111",
  "enrollment": "sMx",
  "deltaDays": -30,
  "deltaHours": -1,
  "hourOfDay": 9,
  "minuteOfHour": 10,
  "isWeekend": 0,
  "book_date":"2023-12-20" ,
  "native_language":"es",
  "class_type":"group" 
}
payload = "{ \"dow\": 3, \"ageGroup\": 2, \"studLevel\": 3, \"stuH\": \"2-10\", \"country_iso\": \"AR\", \"enrollment\": \"s07\", \"deltaDays\": -1, \"deltaHours\": -16, \"hourOfDay\": 3, \"minuteOfHour\": 0, \"isWeekend\": 0, \"isHoliday\": 0, \"native_language\": \"es\", \"class_type\": \"orientation\", \"book_date\": \"2023-11-30\" }"
invoke_lambda(payload)

ClientError: An error occurred (AccessDeniedException) when calling the Invoke operation: User: arn:aws:sts::719805443631:assumed-role/bi_sagemaker/SageMaker is not authorized to perform: lambda:InvokeFunction on resource: arn:aws:lambda:us-east-1:719805443631:function:bi_ml_oejr-prob because no identity-based policy allows the lambda:InvokeFunction action

In [ ]:
%%time
import time
import numpy as np
print("Testing")

pc_times = []
non_pc_times = []
start = time.time()
# ~50 minutes
for i in range(15):
    time.sleep(1)
    print("Request:", i)
    start_no_pc = time.time()
    
    response1 = runtime_client.invoke_endpoint(
        EndpointName='bi-oead-prob-sm-xgboost',
        Body='2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1',
        ContentType="text/csv",
    )
    print("---Prob:", response1["Body"].read())
    response2 = runtime_client.invoke_endpoint(
        EndpointName='bi-oead-prob-sm-xgboost',
        Body='2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-7,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1',
        ContentType="text/csv",
    )
    print("---Prob:", response2["Body"].read())
    response3 = runtime_client.invoke_endpoint(
        EndpointName='bi-oead-prob-sm-xgboost',
        Body='1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1',
        ContentType="text/csv",
    )
    print("---Prob:", response3["Body"].read())
    
    end_no_pc = time.time() - start_no_pc
    
    print("Invokation in ms: {}".format((end_no_pc * 1000 )))
    non_pc_times.append(end_no_pc)

non_pc_cold_start = np.mean(non_pc_times)

print("On Demand Serverless Inference Average Cold Start: {}".format(non_pc_cold_start * 1000))
total = time.time() - start
print("Total:",total * 1000)

Testing
Request: 0
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 34.1336727142334
Request: 1
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 25.240182876586914
Request: 2
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 27.0230770111084
Request: 3
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 30.99536895751953
Request: 4
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 26.601791381835938
Request: 5
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 22.354602813720703
Request: 6
---Prob: b'0.9689412713050842'
---Prob: b'0.9807521104812622'
---Prob: b'0.9700378775596619'
Invokation in ms: 25.303363800048828
Request: 7

Predict with pipeline method

In [ ]:
import os
import boto3
import json
#import pandas as pd
import pickle
from io import BytesIO
import numpy as np
#import sagemaker


#sess = sagemaker.Session()
prefix = 'oead-prob' # use this prefix to store all files
key = prefix + '/model/' + pipeline_name

#load pipeline
s3 = boto3.resource('s3')
bucket_str = bucket
bucket_key = key
with BytesIO() as data:
    s3.Bucket(bucket_str).download_fileobj(bucket_key, data)
    data.seek(0)    # move back to the beginning after writing
    pipeline = pickle.load(data)

    
runtime_client = boto3.client('runtime.sagemaker')

def lambda_handler(event, context):
    
    #df_predict = pd.DataFrame([event_data])
    data_transformed = pipeline.named_steps['preprocessor'].transform([event_data]).toarray()
    df_predict = pd.DataFrame(data_transformed)
    data = df_predict.to_csv(header=False,index=False)
     
    
    payload = data
    #print("payload:" + payload)  
    response = runtime_client.invoke_endpoint(EndpointName=endpoint_name,
                                       ContentType='text/csv',
                                       Body=payload)
    #print(response)
    result = json.loads(response['Body'].read().decode())
    
    print("Pipeline result:" , pipeline.named_steps['classifier'].predict(data_transformed))
    print("Pipeline result:" , pipeline.named_steps['classifier'].predict_proba(data_transformed))
    
    return result

lambda_handler(event_data,0)

workarea

In [ ]:
import os
import json
import pandas as pd
import pickle
from io import BytesIO
import datetime

runtime_client = boto3.client('runtime.sagemaker')
EndpointName='bi-oead-prob-sm-xgboost'
Preprocess='preprocessing.csv'
#sess = sagemaker.Session()
bucket= sess.default_bucket()
prefix = 'oead-prob' # use this prefix to store all files
modelprefix = prefix + '/model/'
dataprefix = prefix + '/train_data/'
sess = sagemaker.Session()

event = {

  "ageGroup": 2,
  "studLevel": 3,
  "stuH": "4-8",
  "country_iso": "AR",
  "enrollment": "sMx",
  "deltaDays": 0,
  "deltaHours": -1,
#  "dow": 3,
#  "hourOfDay": 9,
#  "minuteOfHour": 30,
#  "isWeekend": 0,
  "book_date":"2023-12-20 09:30:00" ,
  "native_language":"es",
  "class_type":"group" 
}

df_holiday = pd.read_csv('s3://{}/{}'.format(rawbucket,dataprefix)+'lk_holiday.csv')

def lambda_handler(event, context):
    #load preprocess template
    preprocess_template_path = 's3://{}/{}'.format(bucket,modelprefix) 
    #print(preprocess_template_path)
    preprocessing_load = pd.read_csv(preprocess_template_path+Preprocess)
    #preprocessing_load = preprocessing_load.drop(['Unnamed: 0'],axis=1)
    
    df_categorical = pd.DataFrame()
    #categorical
    categorical_features = ['stuH', 'country_iso', 'enrollment','native_language','class_type']
    remainder_features = ['deltaDays', 'deltaHours','dow','hourOfDay','minuteOfHour','isWeekend', 'ageGroup', 'studLevel',
                          'isHoliday','isHolidayPre','isHolidayPost']
    
    
    
    #date preprocess
    book_datetime = datetime.datetime.strptime(event['book_date'], "%Y-%m-%d %H:%M:%S")
    event['dow'] = book_datetime.isoweekday()
    event['hourOfDay'] = book_datetime.hour
    event['minuteOfHour'] = book_datetime.minute
    if 6 <= dow <= 7: 
            event['isWeekend'] =  1 
    else: 
            event['isWeekend'] =  0
    
    
    
    #holidays
    event['isHoliday']=0
    event['isHolidayPre']=0
    event['isHolidayPost']=0
    
    #holiday = df_holiday.loc[(df_holiday['date'] == event['book_date']) & (df_holiday['country_code'] == event['country_iso'])]
    holiday = df_holiday.loc[(df_holiday['date'] == book_datetime.strftime("%Y-%m-%d")) & (df_holiday['country_code'] == event['country_iso'])]
    if holiday.empty==False:
        event['isHoliday']=1
    else:
        event['isHoliday']=0
    
    
    #date_post=datetime.datetime.strptime(event['book_date'], "%Y-%m-%d")+ datetime.timedelta(days=1)
    date_post = book_datetime + datetime.timedelta(days=1)
    date_post=df_holiday.loc[(df_holiday['date'] == date_post.strftime("%Y-%m-%d")) & (df_holiday['country_code'] == event['country_iso'])].date

    if date_post.empty==False:
        event['isHolidayPost']=1
    else:
        event['isHolidayPost']=0

    #date_pre=datetime.datetime.strptime(event['book_date'], "%Y-%m-%d")+ datetime.timedelta(days=-1)
    date_pre= book_datetime + datetime.timedelta(days=-1)
    date_pre=df_holiday.loc[(df_holiday['date'] == date_pre.strftime("%Y-%m-%d")) & (df_holiday['country_code'] == event['country_iso'])].date

    if date_pre.empty==False:
        event['isHolidayPre']=1
    else:
        event['isHolidayPre']=0
    
    
    
    del event['book_date']
    
    event['isWeekend'] = int(event['isWeekend'] == 'true')
    event = pd.DataFrame([event])

    event_ohe = event[categorical_features].copy()
    event_remainder = event[remainder_features].copy()

    #create dummy variable for each categorical feature
    for x in categorical_features:
        df = pd.get_dummies(event_ohe[[x]],columns=[str(x)],prefix=str(x)) 
        event_ohe.drop(columns=[x],axis=1,inplace=True)
        df_categorical = df.join(df_categorical)


    #combine categorical features with ohe and remainder features in 1 dataframe
    event_preprocessed = event_remainder.combine_first(df_categorical.astype(int))
    event_preprocessed = event_preprocessed.reset_index(drop=True)
    #event_preprocessed.to_csv('event_preprocessed.csv', index=False)
    #combine even_preprocessed with the rest of ohe columns from trainning dataset
    #predict = event_preprocessed.join(preprocessing_load.drop(columns=event_preprocessed.columns,axis=1))
    
    predict = preprocessing_load.merge(event_preprocessed, how='right')
    predict = predict.fillna(0)
    predict.to_csv('predict.csv')
    data = predict.to_csv(header=False,index=False)
    
    #payload = '2,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,-1,6,0.0,0.0,0.0,0.0,0.0,0.0,1,11,1,0,0,0,30,1,0.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1'
    payload='2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,6,0,0,0,0,0,0,1,11,1,0,0,0,30,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1'
    #payload = data
    #print("payload:" + payload)  
    response = runtime_client.invoke_endpoint(EndpointName=EndpointName,
                                       ContentType='text/csv',
                                       Body=payload)
    #print(response)
    result = json.loads(response['Body'].read().decode())
    #result = response['Body'].read().decode('ascii')
    #result=0
    return result,data,event

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


In [107]:
result,data,event=lambda_handler(event,0)

In [108]:
result

0.9689412713050842

In [109]:
data

'2,0.0,1,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,-1,3,0.0,0.0,0.0,0.0,0.0,0.0,1,9,0,0,0,0,30,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,3\n'

In [110]:
event

,ageGroup,studLevel,stuH,country_iso,enrollment,deltaDays,deltaHours,native_language,class_type,dow,hourOfDay,minuteOfHour,isWeekend,isHoliday,isHolidayPre,isHolidayPost
0,2,3,4-8,AR,sMx,0,-1,es,group,3,9,30,0,0,0,0


In [15]:
evento={
    "dow": 1,
    "ageGroup": 1,
    "studLevel": 1,
    "stuH": "",
    "country_iso": "AR",
    "enrollment": "s31",
    "deltaDays": 0,
    "deltaHours": -4,
    "hourOfDay": 14,
    "minuteOfHour": 0,
    "isWeekend": 'true',
    "native_language": "es",
    "class_type": "group",
    "book_date": "2024-01-29",
    "salesforce_contact_id": "003DV00001m00QPYAY"
}

In [16]:
evento['isWeekend']

'true'

In [17]:
evento['isWeekend'] = int(evento['isWeekend'] == 'true')

In [18]:
evento['isWeekend']

1